In [5]:
import numpy as np
import matplotlib.pyplot as plt
from hashlib import sha256

# ---------- 1. Turn a SHA‑256 hex digest into a 2‑D “turbulence” field ----------
def hash_to_field(hex_digest: str, width: int = 128) -> np.ndarray:
    """
    Map the 64‑hex‑char SHA‑256 digest to a 2‑D numeric field.
    Each hex digit (0‑F) is expanded to its integer value 0‑15.
    The sequence is wrapped row‑by‑row into a matrix of the given width.
    Missing cells (if any) are padded with zeros.
    """
    values = [int(ch, 16) for ch in hex_digest.strip()]
    # repeat the sequence so we have at least width*height samples
    reps = (width * ((len(values) + width - 1) // width)) // len(values) + 1
    values = (values * reps)[: width * ((len(values) + width - 1) // width)]
    field = np.array(values, dtype=float).reshape(-1, width)
    return field


# ---------- 2. Utility: compute the local gradient (negative = “down‑hill”) ----------
def field_gradient(field: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    dy, dx = np.gradient(field)
    return -dx, -dy   # negative gradient → descend toward minima


# ---------- 3. Plinko probe simulator --------------------------------------------
def drop_probes(field: np.ndarray, n_probes: int = 40, max_steps: int = 250,
                step_size: float = 0.4) -> list[np.ndarray]:
    """
    Drop n_probes particles at random x along the top row.
    Each probe moves step_size along the (bilinear‑interpolated) negative gradient
    until it exits the field or reaches max_steps.
    Returns a list of (N_i,2) arrays with x,y coordinates of the path.
    """
    h, w = field.shape
    gx, gy = field_gradient(field)

    # quick bilinear sampler for the gradient
    def sample_grad(xf: float, yf: float) -> tuple[float, float]:
        if xf < 0 or yf < 0 or xf > w - 2 or yf > h - 2:
            return (0.0, 0.0)
        x0, y0 = int(xf), int(yf)
        dx1, dy1 = xf - x0, yf - y0
        weights = np.array(
            [[(1-dx1)*(1-dy1), dx1*(1-dy1)],
             [(1-dx1)*dy1,      dx1*dy1  ]])
        gx_block = gx[y0:y0+2, x0:x0+2]
        gy_block = gy[y0:y0+2, x0:x0+2]
        return (float((weights*gx_block).sum()),
                float((weights*gy_block).sum()))

    paths = []
    rng = np.random.default_rng(123)
    for _ in range(n_probes):
        x, y = rng.uniform(0, w-1), 0.0
        pts = [(x, y)]
        for _ in range(max_steps):
            vx, vy = sample_grad(x, y)
            norm = (vx**2 + vy**2) ** 0.5 + 1e-9
            x += step_size * vx / norm
            y += step_size * vy / norm
            if x < 0 or x >= w or y < 0 or y >= h:
                break
            pts.append((x, y))
        paths.append(np.array(pts))
    return paths


# ---------- 4. Quick demo --------------------------------------------------------
# a real SHA‑256 hex digest (you can replace with any 64‑char string)
digest = sha256(b"Hello, π‑carrier!").hexdigest()
field   = hash_to_field(digest, width=128)
paths   = drop_probes(field, n_probes=60, max_steps=300)

# ---------- 5. Plot heat‑map + probe trajectories + drift vectors ---------------
fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(field, cmap="viridis", origin="upper", interpolation="nearest")
fig.colorbar(im, ax=ax, label="Field value (0–15)")

# overlay a sparse vector field (every 8th cell) for visual clarity
gdx, gdy = field_gradient(field)
stride = 8
Y, X = np.mgrid[0:field.shape[0]:stride, 0:field.shape[1]:stride]
ax.quiver(X, Y, gdx[::stride, ::stride], gdy[::stride, ::stride],
          color="white", alpha=0.6, scale=100, width=0.002)

# plot probe paths
for p in paths:
    ax.plot(p[:,0], p[:,1], color="red", alpha=0.6, linewidth=1)

ax.set_title("SHA‑256 Harmonic Field • Drift Vectors • Plinko Probe Paths")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_xlim(0, field.shape[1]-1)
ax.set_ylim(field.shape[0]-1, 0)   # invert y‑axis so 0 is top

plt.tight_layout()



SyntaxError: bytes can only contain ASCII literal characters (2169420234.py, line 72)

In [3]:
import numpy as np
import matplotlib.pyplot as plt

# ─── 1-A.  normalise the matrix you already built ────────────────────────────
field  = matrix.astype(float)                # ‘matrix’ came from plot_lean()
field -= field.min()
field /= field.max() + 1e-9                  # 0‥1 range  (small eps to avoid /0)

# ─── 1-B.  quick tension-gradient utility (finite differences) ───────────────
def tension_vectors(F):
    """Return x- and y- gradients (downhill = positive ‘pull’)."""
    gx = np.zeros_like(F)
    gy = np.zeros_like(F)
    gx[:, :-1] = F[:, :-1] - F[:, 1:]        # higher → lower  (eastward diff)
    gy[:-1, :] = F[:-1, :] - F[1:, :]        #   ”      ”      (southward diff)
    return gx, gy
GX, GY = tension_vectors(field)
# ─── 2-A.  parameters you can tweak easily ───────────────────────────────────
N_PROBES   = 250         # how many particles to drop
N_STEPS    = 250         # max steps per probe
STEP       = 1.0         # pixel movement per step
DROP_EDGE  = "top"       # "top", "bottom", "left", or "right"

h, w = field.shape
paths = []               # will hold a list of (x[], y[]) for each probe

# ─── 2-B.  helper to pick random spawn coord on chosen edge ──────────────────
def random_spawn(edge):
    if edge == "top":    return np.random.randint(0, w), 0
    if edge == "bottom": return np.random.randint(0, w), h - 1
    if edge == "left":   return 0, np.random.randint(0, h)
    if edge == "right":  return w - 1, np.random.randint(0, h)
    raise ValueError("edge must be top / bottom / left / right")

# ─── 2-C.  main probe loop ───────────────────────────────────────────────────
for _ in range(N_PROBES):
    x, y  = random_spawn(DROP_EDGE)
    xs, ys = [x], [y]

    for _ in range(N_STEPS):
        # sample gradient; if flat region, break
        gx, gy = GX[int(y) % h, int(x) % w], GY[int(y) % h, int(x) % w]
        gnorm  = np.hypot(gx, gy)
        if gnorm < 1e-6:      # almost zero tension → particle rests
            break
        # move downhill (steepest descent)
        x += (gx / gnorm) * STEP
        y += (gy / gnorm) * STEP
        xs.append(x); ys.append(y)

        # stop if wandered outside
        if not (0 <= x < w and 0 <= y < h):
            break
    paths.append((xs, ys))
plt.figure(figsize=(12, 6))
plt.imshow(field, cmap="viridis", origin="upper", interpolation="nearest")
plt.colorbar(label="Normalised Field Intensity")

# overlay drift (sample every N pixels for clarity)
skip = 4
plt.quiver(np.arange(0, w, skip), 
           np.arange(0, h, skip),
           GX[::skip, ::skip], 
           GY[::skip, ::skip],
           color="white",  linewidth=0.4, alpha=0.6)

# overlay Plinko probe paths
for xs, ys in paths:
    plt.plot(xs, ys, lw=0.7, alpha=0.6)

plt.title("SHA-256 Harmonic Expansion | Tension Field, Drift & Plinko Echoes")
plt.axis('off')
plt.show()


NameError: name 'matrix' is not defined